# ASHA Sahayak — GRPO Training Notebook

Training an LLM to assist ASHA (Accredited Social Health Activist) workers in rural India 
with clinical triage decisions using Group Relative Policy Optimization (GRPO).

**Environment**: ASHA Sahayak OpenEnv RL environment  
**Algorithm**: GRPO (Group Relative Policy Optimization) via TRL  
**Base model**: Qwen/Qwen3-0.6B (fits on Colab free T4 GPU)  
**Ground truth**: Indian Government IMNCI protocol  

## Why GRPO?

Each clinical case is a multi-turn episode. The agent asks clarifying questions 
(exploration) then makes a referral decision (exploitation). GRPO trains the model 
to learn which questions matter and which referral is correct — exactly like 
Savitri needs to do at 2 AM in Sitapur district.


In [ ]:
# Install dependencies
!pip install -q "trl>=0.15.0" "transformers>=4.40.0" "httpx>=0.24.0" "pydantic>=2.0.0" "matplotlib" "unsloth"


In [ ]:
import os

# Set these before running
ENV_BASE_URL = os.getenv("ENV_BASE_URL", "https://sreenathmmenon-asha-sahayak.hf.space")
HF_TOKEN = os.getenv("HF_TOKEN", "")  # Your HuggingFace token

# Training config
BASE_MODEL = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "asha-sahayak-grpo"
NUM_TRAIN_STEPS = 200
LEARNING_RATE = 5e-6
NUM_GENERATIONS = 4

print(f"ENV URL: {ENV_BASE_URL}")
print(f"Base model: {BASE_MODEL}")
print(f"Training steps: {NUM_TRAIN_STEPS}")


In [ ]:
import httpx
import json
from typing import Optional

class AshaClient:
    """HTTP client for ASHA Sahayak OpenEnv environment."""
    
    def __init__(self, base_url: str = ENV_BASE_URL):
        self.base_url = base_url.rstrip("/")
        self.session_id: Optional[str] = None
    
    def reset(self, task_id: str = "easy", seed: int = 42) -> dict:
        resp = httpx.post(
            f"{self.base_url}/reset",
            json={"task_id": task_id, "seed": seed},
            timeout=30.0
        )
        resp.raise_for_status()
        data = resp.json()
        self.session_id = data.get("session_id")
        return data
    
    def step(self, referral_decision: str = "PENDING", urgency: str = "monitor",
             primary_concern: str = "", question: Optional[str] = None,
             confidence: float = 0.8) -> dict:
        headers = {}
        if self.session_id:
            headers["X-Session-ID"] = self.session_id
        resp = httpx.post(
            f"{self.base_url}/step",
            json={
                "referral_decision": referral_decision,
                "urgency": urgency,
                "primary_concern": primary_concern,
                "question": question,
                "confidence": confidence,
            },
            headers=headers,
            timeout=30.0,
        )
        resp.raise_for_status()
        return resp.json()["observation"]

# Test the connection
client = AshaClient()
result = client.reset(task_id="easy", seed=42)
print("Connection OK!")
print("Initial case:", result["observation"]["conversation"][0]["text"][:200])


In [ ]:
class AshaToolEnv:
    """
    TRL-compatible environment for ASHA Sahayak.
    
    Tools are discovered from public methods with typed args and docstrings.
    TRL calls reset() at episode start, then calls tool methods as the model requests.
    The reward is read from self.reward after episode completion.
    """
    
    def __init__(self):
        self.client = AshaClient(base_url=ENV_BASE_URL)
        self.reward: float = 0.0
        self._done: bool = False
        self._task_id: str = "easy"
    
    def reset(self, task_id: str = "easy", seed: int = 42) -> str:
        """Reset the environment. Called automatically by TRL at episode start."""
        self.reward = 0.0
        self._done = False
        self._task_id = task_id
        data = self.client.reset(task_id=task_id, seed=seed)
        obs = data["observation"]
        return obs["conversation"][0]["text"]
    
    def ask_question(self, question: str) -> str:
        """Ask the ASHA worker a clinical clarifying question to gather more information.

        Args:
            question: A clinical question to ask, e.g. 'Does the child have fast breathing?'
                     or 'Is there chest indrawing?' or 'When did the fever start?'

        Returns:
            The ASHA worker's response describing what they observe in the patient.
        """
        if self._done:
            return "Episode already complete. Call reset() to start a new episode."
        obs = self.client.step(
            referral_decision="PENDING",
            urgency="monitor",
            primary_concern="",
            question=question,
            confidence=0.5,
        )
        self.reward = obs.get("reward", 0.0)
        self._done = obs.get("done", False)
        if obs.get("conversation"):
            return obs["conversation"][-1]["text"]
        return "No response received."
    
    def make_referral(
        self,
        referral_decision: str,
        urgency: str,
        primary_concern: str,
        confidence: float = 0.8,
    ) -> str:
        """Make the final clinical referral decision for the patient.

        Args:
            referral_decision: One of REFER_IMMEDIATELY, REFER_WITHIN_24H, TREAT_AT_HOME, MONITOR
            urgency: One of immediate, within_24h, routine, monitor
            primary_concern: The primary clinical concern identified, e.g. severe_pneumonia
            confidence: Your confidence in this decision, between 0.0 and 1.0

        Returns:
            Clinical feedback with score breakdown showing how correct the decision was.
        """
        if self._done:
            return "Episode already complete."
        obs = self.client.step(
            referral_decision=referral_decision,
            urgency=urgency,
            primary_concern=primary_concern,
            question=None,
            confidence=confidence,
        )
        self.reward = obs.get("reward", 0.0)
        self._done = obs.get("done", False)
        feedback = obs.get("feedback", f"Score: {self.reward:.3f}")
        return feedback or f"Final score: {self.reward:.3f}"

# Smoke test
env = AshaToolEnv()
initial = env.reset(task_id="easy", seed=42)
print("Initial presentation:", initial[:300])

response = env.ask_question("Does the child have fast breathing or chest indrawing?")
print("\nASHA Worker response:", response[:300])

feedback = env.make_referral("REFER_IMMEDIATELY", "immediate", "severe_pneumonia", 0.9)
print("\nFeedback:", feedback[:300])
print("\nFinal reward:", env.reward)


In [ ]:
import random
import numpy as np
from tqdm.auto import tqdm

# Measure baseline reward with random policy
def run_random_baseline(n_episodes: int = 30, task_id: str = "easy") -> list:
    """Run random policy to establish baseline reward."""
    REFERRALS = ["REFER_IMMEDIATELY", "REFER_WITHIN_24H", "TREAT_AT_HOME", "MONITOR"]
    URGENCIES = ["immediate", "within_24h", "routine", "monitor"]
    CONCERNS = ["severe_pneumonia", "pre_eclampsia", "dehydration", "malaria", "sepsis", "tb", "malnutrition"]
    
    rewards = []
    env = AshaToolEnv()
    
    for i in tqdm(range(n_episodes), desc="Baseline episodes"):
        try:
            seed = random.randint(0, 10000)
            env.reset(task_id=task_id, seed=seed)
            # Random policy: ask 1-2 random questions, then make random decision
            for _ in range(random.randint(0, 2)):
                env.ask_question(random.choice([
                    "Does the child have fever?",
                    "How long has the patient been sick?",
                    "Is there any bleeding?",
                    "Can the patient drink fluids?",
                ]))
                if env._done:
                    break
            if not env._done:
                env.make_referral(
                    random.choice(REFERRALS),
                    random.choice(URGENCIES),
                    random.choice(CONCERNS),
                    confidence=random.uniform(0.3, 0.9)
                )
            rewards.append(env.reward)
        except Exception as e:
            print(f"Episode {i} error: {e}")
            rewards.append(0.1)
    
    return rewards

print("Measuring random policy baseline...")
baseline_easy = run_random_baseline(n_episodes=30, task_id="easy")
baseline_medium = run_random_baseline(n_episodes=20, task_id="medium")
baseline_hard = run_random_baseline(n_episodes=20, task_id="hard")

print(f"\nBaseline Results (Random Policy):")
print(f"  Easy:   {np.mean(baseline_easy):.3f} \u00b1 {np.std(baseline_easy):.3f}")
print(f"  Medium: {np.mean(baseline_medium):.3f} \u00b1 {np.std(baseline_medium):.3f}")
print(f"  Hard:   {np.mean(baseline_hard):.3f} \u00b1 {np.std(baseline_hard):.3f}")
print(f"  Overall: {np.mean(baseline_easy + baseline_medium + baseline_hard):.3f}")


In [ ]:
from datasets import Dataset

# Create training prompts — GRPO will call env.reset(**row) for each
training_data = []

# 200 easy episodes
for seed in range(100):
    training_data.append({"task_id": "easy", "seed": seed})

# 150 medium episodes  
for seed in range(75):
    training_data.append({"task_id": "medium", "seed": seed})

# 100 hard episodes
for seed in range(50):
    training_data.append({"task_id": "hard", "seed": seed})

# Shuffle
random.shuffle(training_data)

# System prompt
SYSTEM_PROMPT = """You are an AI assistant helping ASHA (Accredited Social Health Activist) workers in rural India make clinical triage decisions.

Your workflow:
1. GATHER INFORMATION: Ask clarifying questions using ask_question() to understand the patient's condition
2. MAKE DECISION: Use make_referral() with one of:
   - REFER_IMMEDIATELY: Life-threatening emergency, call 108 now
   - REFER_WITHIN_24H: Needs medical attention within 24 hours
   - TREAT_AT_HOME: Can be managed at home with ASHA guidance
   - MONITOR: Observe and follow up

Ask at least 1-2 clarifying questions before making your referral decision. 
Gather information about: fever duration, breathing difficulties, consciousness level, feeding ability, bleeding, skin color.

Ground truth: Indian Government IMNCI (Integrated Management of Neonatal and Childhood Illness) protocol."""

# Format for TRL (prompt column)
dataset_rows = [
    {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}],
        "task_id": row["task_id"],
        "seed": row["seed"],
    }
    for row in training_data
]

dataset = Dataset.from_list(dataset_rows)
print(f"Training dataset: {len(dataset)} episodes")
print(f"  Easy: {sum(1 for r in training_data if r['task_id'] == 'easy')}")
print(f"  Medium: {sum(1 for r in training_data if r['task_id'] == 'medium')}")
print(f"  Hard: {sum(1 for r in training_data if r['task_id'] == 'hard')}")


In [ ]:
from trl import GRPOTrainer, GRPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded!")

# GRPO configuration
config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_generations=NUM_GENERATIONS,
    max_completion_length=1024,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_steps=50,
    report_to="none",  # Set to "wandb" for logging
    max_steps=NUM_TRAIN_STEPS,
)

def environment_factory():
    """Create a new ASHA environment instance for each GRPO rollout."""
    return AshaToolEnv()

def reward_func(env: AshaToolEnv, **kwargs) -> float:
    """Extract reward from completed episode."""
    return env.reward

# Create trainer
trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    environment_factory=environment_factory,
    reward_funcs=reward_func,
    tokenizer=tokenizer,
)

print("Starting GRPO training...")
print("This will take approximately 60-90 minutes on a T4 GPU.")
trainer.train()
print("Training complete!")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Extract rewards from trainer logs
logs = trainer.state.log_history
steps = [l["step"] for l in logs if "train/reward" in l or "reward" in l]
rewards_logged = [l.get("train/reward", l.get("reward", 0)) for l in logs if "train/reward" in l or "reward" in l]

# Plot training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Training reward curve
axes[0].plot(steps, rewards_logged, 'b-', linewidth=2, alpha=0.8, label='Training reward')
if len(rewards_logged) > 5:
    # Smooth with rolling average
    window = max(1, len(rewards_logged) // 10)
    smoothed = np.convolve(rewards_logged, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards_logged)), smoothed, 'r-', linewidth=2.5, label='Smoothed')
axes[0].axhline(y=np.mean(baseline_easy + baseline_medium + baseline_hard), 
                color='gray', linestyle='--', label=f'Random baseline ({np.mean(baseline_easy + baseline_medium + baseline_hard):.3f})')
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Reward', fontsize=12)
axes[0].set_title('ASHA Sahayak — GRPO Training Reward', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1.0)
axes[0].grid(True, alpha=0.3)

# Right: Post-training vs baseline comparison (measure after training)
post_easy = run_random_baseline(n_episodes=20, task_id="easy")  # Will reuse method but policy is trained now
axes[1].bar(['Baseline\n(Random)', 'Trained\n(GRPO)'], 
            [np.mean(baseline_easy), np.mean(post_easy)],
            color=['#ff6b6b', '#51cf66'], edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('Mean Reward', fontsize=12)
axes[1].set_title('Before vs After GRPO Training\n(Easy Cases)', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([np.mean(baseline_easy), np.mean(post_easy)]):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('asha_grpo_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

improvement = np.mean(post_easy) - np.mean(baseline_easy)
print(f"\nTraining Results:")
print(f"  Baseline (random): {np.mean(baseline_easy):.3f}")
print(f"  After GRPO:        {np.mean(post_easy):.3f}")
print(f"  Improvement:       +{improvement:.3f} ({improvement/np.mean(baseline_easy)*100:.1f}%)")


In [ ]:
# Save the trained model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")

# Optionally push to HF Hub
if HF_TOKEN:
    trainer.push_to_hub(
        f"sreenathmmenon/asha-sahayak-grpo",
        token=HF_TOKEN,
    )
    print("Model pushed to HuggingFace Hub!")


In [ ]:
# Demonstrate multi-agent episode with trained model
print("=" * 60)
print("MULTI-AGENT DEMO: ASHA Worker + PHC Doctor")
print("=" * 60)

import httpx, json

BASE = ENV_BASE_URL

# Phase 1: Multi-agent reset
r = httpx.post(f"{BASE}/multi/reset", json={"task_id": "medium", "seed": 42}, timeout=30)
data = r.json()
session_id = data["session_id"]
print(f"\nPhase 1 — ASHA Worker")
print(f"Session: {session_id}")
print(f"Case: {data['observation']['conversation'][0]['text'][:200]}")

# ASHA asks a question
r = httpx.post(f"{BASE}/multi/step/asha",
    headers={"X-Session-ID": session_id},
    json={"referral_decision": "PENDING", "urgency": "monitor", "primary_concern": "",
          "question": "How long has the child been coughing, and is there any evening fever?"},
    timeout=30)
obs = r.json()
print(f"\nASHA Question \u2192 {obs['observation']['conversation'][-1]['text'][:200]}")

# ASHA makes final decision
r = httpx.post(f"{BASE}/multi/step/asha",
    headers={"X-Session-ID": session_id},
    json={"referral_decision": "REFER_WITHIN_24H", "urgency": "within_24h",
          "primary_concern": "pediatric_tb_contact", "question": None, "confidence": 0.88},
    timeout=30)
data = r.json()
print(f"\nASHA Final Decision: REFER_WITHIN_24H | Score: {data.get('asha_score', 'N/A')}")
print(f"\nReferral Note sent to PHC Doctor:")
print(data.get("referral_note", "")[:400])

# Phase 2: PHC Doctor
print(f"\nPhase 2 — PHC Doctor")
r = httpx.post(f"{BASE}/multi/step/doctor",
    headers={"X-Session-ID": session_id},
    json={"disposition": "manage_at_phc",
          "investigations": ["chest_xray", "mantoux_test"],
          "rationale": "Pediatric TB contact with 3-week cough. PHC DOTS program can manage."},
    timeout=30)
result = r.json()
print(f"\nDoctor Disposition: manage_at_phc")
print(f"Combined Episode Reward: {result.get('reward', 'N/A'):.4f}")
print(f"\nBreakdown:")
for k, v in result.get("breakdown", {}).items():
    print(f"  {k}: {v}")
